In [1]:
function range(n: number): number[] {
    return Array.from({ length: n }, (_, i) => i);
}

In [2]:
range(5)

[ 0, 1, 2, 3, 4 ]


# Sudoku

The sudoku we want to solve is shown here:
    <img src="sudoku.png" width="40%">
I have taken it from https://sudoku.zeit.de/sudoku-hoellisch.

The function `create_puzzle` returns a representation of this puzzle as an array of arrays.

In [3]:
type CellValue = number | "*";

function create_puzzle(): CellValue[][] {
    return [ ["*",  3 ,  9 , "*", "*", "*", "*", "*",  7 ], 
             ["*", "*", "*",  7 , "*", "*",  4 ,  9 ,  2 ],
             ["*", "*", "*", "*",  6 ,  5 , "*",  8 ,  3 ],
             ["*", "*", "*",  6 , "*",  3 ,  2 ,  7 , "*"],
             ["*", "*", "*", "*",  4 , "*",  8 , "*", "*"],
             [ 5 ,  6 , "*", "*", "*", "*", "*", "*", "*"],
             ["*", "*",  5 ,  2 , "*",  9 , "*", "*",  1 ],
             ["*",  2 ,  1 , "*", "*", "*", "*",  4 , "*"],
             [ 7 , "*", "*", "*", "*", "*",  5 , "*", "*"]
           ];
}

We are going to solve this puzzle with the help of the constraint solver `Z3`.

In [4]:
import { init, Arith, Bool } from 'z3-solver';
const { Context } = await init();
const Z3 = Context("main");

The function `constraints_from_puzzle` takes one argument:
* `Variables` is a matrix of `Z3` variables.  

   For `row`$\in \{0,8\}$ and `col`$\in \{0,8\}$ the variable `Variables[row][col]` specifies the number that is placed in the specified row and column.

It returns a set of constraints specifying that the variables corresponding to numbers that are already set in the given Sudoku take the specified values.

In [6]:
function constraints_from_puzzle(Variables: Arith[][]): Bool[] {
    const Puzzle = create_puzzle();
    return range(9).flatMap(y => range(9).filter(x => Puzzle[y][x] != "*").map(x => Variables[y][x].eq(Puzzle[y][x] as number)));
}

In [7]:
const Variables: Arith[][] = 
    range(9).map(row => range(9).map(col => Z3.Int.const(`V${row + 1}${col + 1}`)));

The function `all_constraints` returns a CSP that encodes the given sudoku as a CSP.

In [9]:
function all_constraints(Variables: Arith[][]): Bool[] {
    const constraints    = constraints_from_puzzle(Variables);
    const rowConstraints = range(9).map(row => Z3.Distinct(...Variables[row]));
    const colConstraints = range(9).map(col => Z3.Distinct(...range(9).map(row => Variables[row][col])));
    const squareConstraints = range(3).flatMap(bRow => range(3).map(bCol =>
                                Z3.Distinct(...range(3).flatMap(row => range(3).map(col => Variables[3*bRow + row][3*bCol + col])))));

    const boundConstraints = range(9).flatMap(row => range(9).map(col => Z3.And(Variables[row][col].ge(1), Variables[row][col].le(9))));
    
    return [
        ...constraints,
        ...rowConstraints,
        ...colConstraints,
        ...squareConstraints,
        ...boundConstraints
    ];
}

I have got 217 constraints.

In [10]:
all_constraints(Variables).length

136


The function `solve()` computes a solution to the given problem and returns this solution.

In [11]:
import { RecursiveMap as Map } from 'recursive-set';

async function solve(): Promise<Map<string, number> | undefined> {
    const Variables: Arith[][] = range(9).map(row => 
        range(9).map(col => Z3.Int.const(`V${row + 1}${col + 1}`))
    );
    
    const S = new Z3.Solver();
    S.add(...all_constraints(Variables));
    
    const result = await S.check();
    if (result == 'sat') {
        const M = S.model();
        const Solution = new Map<string, number>();
        range(9).forEach(row => 
            range(9).forEach(col => {
                const varName = `V${row + 1}${col + 1}`;
                const valStr = M.eval(Variables[row][col]).toString();
                Solution.set(varName, parseInt(valStr, 10));
            })
        );
        return Solution;
    } else if (result === 'unsat') {
        console.log('The problem is not solvable.');
    } else {
        console.log('Z3 cannot determine whether the problem is solvable.');
    }
}

In [12]:
console.time("Solver Duration");
const Solution = await solve();
console.timeEnd("Solver Duration");
Solution

Solver Duration: 1.546s
RecursiveMap(81) {
  V11 => 1,
  V12 => 3,
  V13 => 9,
  V14 => 4,
  V15 => 2,
  V16 => 8,
  V17 => 6,
  V18 => 5,
  V19 => 7,
  V21 => 6,
  V22 => 5,
  V23 => 8,
  V24 => 7,
  V25 => 3,
  V26 => 1,
  V27 => 4,
  V28 => 9,
  V29 => 2,
  V31 => 2,
  V32 => 4,
  V33 => 7,
  V34 => 9,
  V35 => 6,
  V36 => 5,
  V37 => 1,
  V38 => 8,
  V39 => 3,
  V41 => 8,
  V42 => 1,
  V43 => 4,
  V44 => 6,
  V45 => 9,
  V46 => 3,
  V47 => 2,
  V48 => 7,
  V49 => 5,
  V51 => 9,
  V52 => 7,
  V53 => 3,
  V54 => 5,
  V55 => 4,
  V56 => 2,
  V57 => 8,
  V58 => 1,
  V59 => 6,
  V61 => 5,
  V62 => 6,
  V63 => 2,
  V64 => 1,
  V65 => 8,
  V66 => 7,
  V67 => 9,
  V68 => 3,
  V69 => 4,
  V71 => 4,
  V72 => 8,
  V73 => 5,
  V74 => 2,
  V75 => 7,
  V76 => 9,
  V77 => 3,
  V78 => 6,
  V79 => 1,
  V81 => 3,
  V82 => 2,
  V83 => 1,
  V84 => 8,
  V85 => 5,
  V86 => 6,
  V87 => 7,
  V88 => 4,
  V89 => 9,
  V91 => 7,
  V92 => 9,
  V93 => 6,
  V94 => 3,
  V95 => 1,
  V96 => 4,
  V97 => 5,
  V98 => 

## Graphical Representation

The function `show_solution` displays the given solution on a 9x9 grid, rendering it directly to SVG using `tslab`.

In [13]:
import * as tslab from "tslab";
import { RecursiveMap as Map } from "recursive-set";

function show_solution(Solution: Map<string, number> | undefined, width: string = "50%"): void {
    if (!Solution) {
        console.log("No solution to display.");
        return;
    }
    const Sudoku = create_puzzle();
    const cellSize = 50;
    const totalSize = 9 * cellSize;
    
    let html = `<svg width="${width}" viewBox="-5 -5 ${totalSize + 10} ${totalSize + 10}" xmlns="http://www.w3.org/2000/svg" style="font-family: sans-serif;">`;
    
    html += range(9).flatMap(r => 
        range(9).map(c => {
            const x = c * cellSize;
            const y = r * cellSize;
            const isGiven = Sudoku[r][c] !== '*';
            const val = isGiven ? Sudoku[r][c] : Solution.get(`V${r + 1}${c + 1}`);
            const bg = isGiven ? '#f0f0f0' : '#ffffff';
            const color = isGiven ? '#000000' : '#1a73e8';
            const weight = isGiven ? 'bold' : 'normal';
            
            let cellHtml = `<rect x="${x}" y="${y}" width="${cellSize}" height="${cellSize}" fill="${bg}" stroke="#cccccc" stroke-width="1" />`;
            if (val !== undefined) {
                cellHtml += `<text x="${x + cellSize / 2}" y="${y + cellSize / 2 + 7}" text-anchor="middle" font-size="22" fill="${color}" font-weight="${weight}">${val}</text>`;
            }
            return cellHtml;
        })
    ).join('');
    
    html += [0, 3, 6, 9].map(i => {
        const pos = i * cellSize;
        return `<line x1="0" y1="${pos}" x2="${totalSize}" y2="${pos}" stroke="#333333" stroke-width="3" stroke-linecap="square" />` +
               `<line x1="${pos}" y1="0" x2="${pos}" y2="${totalSize}" stroke="#333333" stroke-width="3" stroke-linecap="square" />`;
    }).join('');
    
    html += `</svg>`;
    tslab.display.html(html);
}

In [14]:
show_solution(Solution);

1 3 9 4 2 8 6 5 7 6 5 8 7 3 1 4 9 2 2 4 7 9 6 5 1 8 3 8 1 4 6 9 3 2 7 5 9 7 3 5 4 2 8 1 6 5 6 2 1 8 7 9 3 4 4 8 5 2 7 9 3 6 1 3 2 1 8 5 6 7 4 9 7 9 6 3 1 4 5 2 8